# 1.3 Your turn: your own chat

Continues from [`01.2-irc-chat.ipynb`](01.2-irc-chat.ipynb) — same pipeline shape, pointed at
your own export instead of the IRC showcase.

The one extra step is anonymisation. `humanize` maps each real name to a stable nickname, so
you can hand in your work without handing over your friends' names.

In [ ]:
import pandas as pd
from IPython.display import display

from goad_toolkit.datatransforms import Pipeline, RegexFeature, TimeFeatures
from notebooktester import param

from wa_analyzer.data import PROCESSED, load_own_chat

`RegexFeature` is imported here, not rewritten. The class you derived in `01.2-irc-chat.ipynb`
is the one `goad_toolkit` ships — same three modes, same `feature`-not-`name` parameter, same
coverage log on `extract`. Writing it once was the exercise; retyping it in every notebook
that needs it is just a copy waiting to drift out of step with the original.

## Where `own` comes from

`load_own_chat()` reads `config.toml`'s `current` key — the parquet file that this notebook
writes at the bottom of a run. The normal loop is: run the preprocessor, run this notebook
once, then set `current` in `config.toml` to the filename it just wrote, so every later
notebook loads it automatically.

If you have more than one export and want to switch between them without touching the config,
pass a filename directly instead: `load_own_chat(filename="whatsapp-20250101-120000.parq")`
loads exactly that file from `data/processed/` and ignores `current` entirely.

If this is your first run and neither exists yet, see the README's **Run the preprocessor**
section before continuing — the next cell errors with that same pointer if there is nothing
to load.

In [ ]:
own = load_own_chat()

REQUIRE_OWN_CHAT = param(True, test=False)
if own is None and REQUIRE_OWN_CHAT:
    raise RuntimeError(
        "No chat of your own yet. See the README's 'Run the preprocessor' section, then set "
        "`current` in config.toml to the file it saves -- or pass it directly as "
        "load_own_chat(filename=...)."
    )
elif own is None:
    # No config.toml and no export -- the path notebooktester's CI run takes, since it
    # checks out a clean repo. A tiny stand-in keeps every cell below written as if `own`
    # is always real data, instead of guarding each one separately.
    own = pd.DataFrame({
        "timestamp": pd.to_datetime(["2024-01-01 09:00", "2024-01-01 09:05", "2024-01-02 20:00"]),
        "author": ["Alex", "Sam", "Alex"],
        "message": ["morning!", "hey, see the link https://example.com?", "yes indeed"],
    })

own.head()

In [ ]:
from wa_analyzer.humanhasher import humanize

anon = {name: humanize(name) for name in own.author.unique()}
own["author"] = own.author.map(anon)
print(f"{len(anon)} authors, anonymised. For example:")
display(own.head(3))

Now the *same pipeline shape* as `01.2-irc-chat.ipynb`, pointed at a different dataframe.
That is the payoff of having written the steps as objects rather than as cells: nothing
changes but the input.

In [ ]:
own_pipeline = Pipeline()
own_pipeline.add(TimeFeatures, column="timestamp")
own_pipeline.add(RegexFeature, name="urls",
                 column="message", pattern=r"https?://\S+",
                 feature="has_url", mode="has")

own_enriched = own_pipeline.apply(own)
display(own_enriched.head())

> **Your turn.** Add at least two more steps to `own_pipeline` above — they do not have to
> be `RegexFeature`. A count (`str.count`), a group membership, a summary statistic, another
> `TimeFeatures`-style calendar column: anything shaped like a `TransformBase` step works.
> Pick things specific to *your* chat: an emoji, a group in-joke, a language you switch into,
> the way one person always signs off.
>
> Chat data is unusually rich for this: a WhatsApp export gives you timestamps with daily and
> weekly rhythms, natural language, and half a dozen distributions with textbook shapes. Most
> of what makes this course work comes from features you extract yourself in this notebook.

In [ ]:
# >>> add at least two more own_pipeline.add(...) steps above this line >>>

your_turn = own_pipeline.apply(own)
new_columns = set(your_turn.columns) - set(own_enriched.columns)

MIN_NEW_FEATURES = param(2, test=0)
assert len(new_columns) >= MIN_NEW_FEATURES, (  # noqa: S101 -- the check *is* the point here
    f"add at least {MIN_NEW_FEATURES} more own_pipeline.add(...) steps above, so this produces "
    f"columns beyond {sorted(own_enriched.columns)}"
)
print(f"new column(s): {sorted(new_columns) or 'none yet -- add steps above'}")
display(your_turn.head())

In [ ]:
from datetime import datetime

outfile = PROCESSED / f"chat-{datetime.now():%Y%m%d-%H%M%S}.parq"
your_turn.to_parquet(outfile, index=False)
print(f"Wrote {outfile}")
print("Put that filename after `current` in config.toml so the other notebooks find it.")

## 1.4 What to write down

Before moving on, you should be able to answer these about your own data. Not in your head —
written down, because lesson 2 starts by assuming them.

1. **What is one row, now?** One message. Say what a message is in your export — does a
   photo count, a system notice, a message that was deleted?
2. **How many rows did you lose, and to what?** Every parse drops something. Name the number
   and the reason.
3. **How many *people* are in your data?** Not messages — people. That number is much
   smaller, and in lesson 2 you will find out how much it matters.
4. **Which features did you add, and what question is each for?** A feature you cannot
   attach to a question is one you will never use.
5. **What is in there that should not be?** Bots, group-admin notices, one person's phone
   posting twice. You do not have to remove them yet. You have to know they are there.

---

**Where this goes next.** The features you built here are the input to everything that
follows: lesson 2 compares them across people, lesson 3 across time, lesson 4 asks what shape
they have, and lesson 5 asks which of them tell people apart. And the habit — enrich before
you model — is the one the machine learning course builds on directly.